# 02-rolling-tfidf

See [project guide](../../README.md) and [data requirements](../../data/README.md) before execution. Workspace: `data/tweets/`. External inputs are not included. Run cells in order; model fitting and network collection are not run during repository checks.


In [ ]:
from pathlib import Path
import sys
import os
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import workspace
os.chdir(workspace('tweets'))


In [ ]:
%pprint
import warnings
# Keep warnings visible when checking the research environment.

In [ ]:
# Imported Libraries
import os, re
import numpy as np 
import pandas as pd
import seaborn as sns
from nltk import word_tokenize

import matplotlib.pyplot as plt
import matplotlib.image as img
from matplotlib.pyplot import MultipleLocator

import time
from datetime import datetime # datetime.datetime

from scipy import stats # 统计
from tqdm import tqdm   # 进度条
from IPython.display import display # 列表展示库 display(df)

import threading # 多线程
from concurrent.futures import ThreadPoolExecutor , as_completed # 线程池

# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import GridSearchCV

# gensim
import gensim
import gensim.corpora as corpora
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel, LdaModel

In [ ]:
# Load Data
os.chdir(workspace('tweets'))  
path = workspace('tweets')

tw_pitt_text2day  = pd.read_csv(path+'Doc1_text2day.csv', index_col='Date')
tw_pitt_VN2day    = pd.read_csv(path+'Doc2_VN2day.csv', index_col='Date')
tw_pitt_N2day     = pd.read_csv(path+'Doc3_N2day.csv', index_col='Date')

original_text2day = pd.read_csv(path+'Doc4_original_text2day.csv', index_col='Date')
original_VN2day   = pd.read_csv(path+'Doc5_original_VN2day.csv', index_col='Date')
original_N2day    = pd.read_csv(path+'Doc6_original_N2day.csv', index_col='Date')

Docs = [ tw_pitt_text2day,  tw_pitt_VN2day,  tw_pitt_N2day, 
         original_text2day, original_VN2day, original_N2day ]

# Word Count

In [ ]:
# Define word count function
# Define word count function
def count_words(wordslst):
    df = pd.DataFrame( {'Word': wordslst, 'Count':np.zeros(len(wordslst))} )
    wordcounts = df.groupby('Word').agg({'Count':np.size}).sort_values(by = 'Count', ascending = False)
    wordcounts['Prop%'] = (wordcounts.Count/len(wordslst))*100
    wordcounts = wordcounts.reset_index()
    
    # sort_values(by=‘ColumnName’, axis=0 , ascending = True , inplace = False, na_position = ‘last’ )
    # the column name of agg must be pre-determined
    # df = df.reset_index() can maintain the original index as a column and add a new index starting from 0

    return wordcounts
def checkWords(word, wordcounts):
    # wordcounts is the df defined from above using count_words( wordslst )     
    return wordcounts[ wordcounts['Word'] == word ]
def display_wordcounts(Type, rank, wordcounts, display=True):
    n = wordcounts.shape[0]
    
    if rank==-1:
        temp = wordcounts
        rank = wordcounts.shape[0]
    elif Type == 'top':
        temp = wordcounts.iloc[:rank,:]
    elif Type == 'bottom':
        temp = wordcounts.iloc[n-rank:,:]
        
    lst = [(temp.iloc[i,0],temp.iloc[i,1],temp.iloc[i,2]) for i in range(rank)]
    
    if display:     
        for i in range(rank):
            print("%-20s\t%-10d\t%.7f"%(lst[i][0],lst[i][1],lst[i][2]))     
    return lst

# add new stopwords
def add_stopwords(wordlst):
    path = workspace('tweets') + 'extra_stopwords_en.txt'   
    with open(path, 'a') as f:
        for word in wordlst:
            f.write('\n'+word)
    f.close()
    return
def update_stopwords():
    from nltk.corpus import stopwords as stpw
    stopwords = []
    path = workspace('tweets')
    with open(path+"stopwords_en.txt", "r") as f1:
        for stopword in f1.readlines():
            stopwords.append(stopword.strip('\n'))
        
    with open(path+"extra_stopwords_en.txt", "r") as f2:
        for stopword in f2.readlines():
            stopwords.append(stopword.strip('\n'))

    stopwords.extend(stpw.words('english'))
    stopwords = set(stopwords)
    return stopwords
def filter_words2(temp):
    # temp = tweet.lower()  
    # fist filter
    temp = [w for w in temp.split(' ') if not w in stopwords]
    temp = [w for w in temp if len(w)>2]
    
    temp = " ".join(word for word in temp)
    return temp

In [ ]:
%%time
all_words  = [' '.join(list(Doc['Content'])).split() for Doc in Docs]
wordcounts = [count_words(x) for x in all_words]

In [ ]:
# Word Count
DocNum   = 6 # 1,2,3,4,5,6
top_prop = 0

temp_counts = wordcounts[DocNum-1]
print(temp_counts[temp_counts['Prop%']>=top_prop]['Prop%'].sum())
print(temp_counts[temp_counts['Prop%']>=top_prop]['Count'].sum())
print()
top = display_wordcounts('top', -1, temp_counts[temp_counts['Prop%']>=top_prop])

In [ ]:
# Generate wordcloud
# Import the wordcloud library
# import matplotlib.pyplot as plt

from wordcloud import WordCloud
from PIL import Image

def drawWordCloud( wordcounts , topNumber , figuresize , filename ): 
    word_frequency = {x[0]:x[1] for x in wordcounts.head(topNumber).values} # 返回二维列表，外层元素为Observation 
    
    if len(filename) == 0 :
        wordcloud = WordCloud(scale=10, background_color="white", 
                              max_words=topNumber, contour_width=3, contour_color='steelblue')
    # scale - pic quality
    else :
        shape = np.array( Image.open(os.path.join(os.getcwd(),filename)) ) # current working path os.getcwd()
        wordcloud = WordCloud(scale=10, background_color="white", 
                              mask=shape, max_words=topNumber)
     
    # Generate a word cloud
    wordcloud.fit_words(word_frequency)
    
    # Visualize the word cloud
    fig = plt.figure( figsize = figuresize ) 
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.show()
    
    return fig

In [ ]:
fig = drawWordCloud( wordcounts[5] , 500 , (40,40) , 'covid.jpg' )
if False :
        fig.savefig('./plot/Doc6_original_N2day.jpg', transparent = False, bbox_inches = 'tight')

# TF-IDF Analysis

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Load Data
os.chdir(workspace('tweets'))  
path = workspace('tweets')

tw_pitt_text2day  = pd.read_csv(path+'Doc1_text2day.csv', index_col='Date')
tw_pitt_VN2day    = pd.read_csv(path+'Doc2_VN2day.csv', index_col='Date')
tw_pitt_N2day     = pd.read_csv(path+'Doc3_N2day.csv', index_col='Date')

original_text2day = pd.read_csv(path+'Doc4_original_text2day.csv', index_col='Date')
original_VN2day   = pd.read_csv(path+'Doc5_original_VN2day.csv', index_col='Date')
original_N2day    = pd.read_csv(path+'Doc6_original_N2day.csv', index_col='Date')

Docs = [ tw_pitt_text2day,  tw_pitt_VN2day,  tw_pitt_N2day, 
         original_text2day, original_VN2day, original_N2day ]

Days = tw_pitt_text2day.index

## Daily TF-IDF

### Functions

In [ ]:
# Notes

# from sklearn.feature_extraction.text import TfidfVectorizer

# step 1 define the document list
corpus = ['This is the first document.',
          'This document is the second document.',
          'And this is the third one.',
          'Is this the first document?']

# step 2 construct the model object
vectorizer = TfidfVectorizer() 

# step 3 fit the model with the document sequence as the form of "corpus" 
X = vectorizer.fit_transform(corpus) 

# step 4 get terms appeared in the whole documnet set in the form of an array
terms = vectorizer.get_feature_names_out() 

# What dose the vectorizer.fit_transform(document_list) return ?
''' A sparse matrix denoting

                 term 0      term 1        term 2     ...
document 0    tf-idf(0,0)  tf-idf(0,1)   tf-idf(0,2)    
document 1    tf-idf(1,0)  tf-idf(1,1)   tf-idf(1,2) 
document 2    tf-idf(2,0)  tf-idf(2,1)   tf-idf(2,2) 
   .
   .
   .

'''

# What is a sparse martix ?
'''
A sparse matrix in python is a matrix whose elements are mostly zero and thus are eliminated by python, 
one can use Matrix[a,b] to access its elements. A matrix that is mnot sparse is a dense matrix

dense = <sparse>.todense() : transform a sparse matrix to dense form
2Dlist = <dense>.tolist()  : transform a dense matrix to 2Dlist form
print(sparse) : can print the sparse matrix

'''

# whod dose the sklearn calculated the tf-idf ?
# https://www.analyticsvidhya.com/blog/2021/11/how-sklearns-tfidfvectorizer-calculates-tf-idf-values/
'''
Let : D be the total document list as 'corpus'
      d be one specific document in D as 'This is the first document.'
      t be one specific term that has appeared in D as 'This'
      n = Total number of documents available
      t = term for which idf value has to be calculated
      df(t) = Number of documents in which the term t appears
 
tf(t,d) = Number of times a term ‘t’ appears in a document d

idf(t,D) = ln[ (1+n) / ( 1 + df(t) ) ] + 1    (default i.e smooth_idf = True)
idf(t,D) = ln[ n / df(t) ] + 1                (when smooth_idf = False)

tf-idf(t,d,D) = tf(t,d)*idf(t,D)

sklearn than normalize the tf-idfs for each line in the sparse matrix:

                                 term 0      term 1        term 2     ...
                document 0    tf-idf(0,0)  tf-idf(0,1)   tf-idf(0,2)    
                document 1    tf-idf(1,0)  tf-idf(1,1)   tf-idf(1,2) 
                document 2    tf-idf(2,0)  tf-idf(2,1)   tf-idf(2,2) 
                   .
                   .
                   .
 
''';

In [ ]:
# Define a function, receiving a dataframe in the form as cbs_date_to_word and returns a dataframe recording the tf-idfs 
# of each appearing terms as a time sequence
'''   
                                    term 0      term 1        term 2     ...
                        day 0    tf-idf(0,0)  tf-idf(0,1)   tf-idf(0,2)  
                        day 1    tf-idf(1,0)  tf-idf(1,1)   tf-idf(1,2)  
                        day 2    tf-idf(2,0)  tf-idf(2,1)   tf-idf(2,2)  
                           .
                           .
                           .
'''
from analysis_utils import track_tfidf
def plot_tfidf(daily_tfidf, figsize=(16, 10), 
                            title='Local Media Daily Average Standardized TF-IDF Trending',
                            title_size=18, tick_size=15):
    
    fig = plt.figure( figsize = (16, 10) ) 
    plt.plot (daily_tfidf, linewidth=2, linestyle='-', color=(0, 95/255, 115/255)); 

    try:
        m = daily_tfidf.iloc[:,0].min()
        M = daily_tfidf.iloc[:,0].max()
    except:
        m = daily_tfidf.min()
        M = daily_tfidf.max()
    step = (M-m)/20
    ytk = np.linspace(m, M, 21)

    plt.title(title, fontproperties='Times New Roman', fontsize=title_size, fontweight="bold");
    plt.xticks(daily_tfidf.index[0::5], fontproperties='Times New Roman', fontsize=tick_size, rotation=90, fontweight="bold");
    plt.yticks(ytk, fontproperties='Times New Roman', size=tick_size, fontweight="bold");
    plt.grid();
    
    return fig
def plot_combined_tfidf(daily_tfidfs_lst, figsize=(16, 10), 
               title='Local Media Daily Average Standardized TF-IDF Trending',
               title_size=18, tick_size=15):
    
    ax = daily_tfidfs_lst.plot(linewidth=2, figsize=figsize, xlabel=''); 

    # xticks 
    xtk = daily_tfidfs_lst.index[0::5]
    xtkpos = range(0,len(daily_tfidfs_lst.index),5)
    plt.xticks(xtkpos, xtk,fontproperties='Times New Roman', 
               fontsize=tick_size, rotation=90, fontweight="bold");
    # yticks
    m = daily_tfidfs_lst.min().min()
    M = daily_tfidfs_lst.max().max()
    step = (M-m)/20
    ytk = np.linspace(m, M, 21)
    plt.yticks(ytk, fontproperties='Times New Roman', 
               size=tick_size, fontweight="bold"); 

    # title, legend & grid    
    plt.title(title, fontproperties='Times New Roman', 
              fontsize=title_size, fontweight="bold");
    plt.legend(prop={'size':tick_size, 'family':'Times New Roman', 'weight':'bold'})
    plt.grid();
    
    fig = ax.get_figure()
    
    return fig

# detect missing days
def detect_Missing_days(index):
    dates = pd.to_datetime(index)
    missing_day = []
    for i in range(1,len(dates)):
        delta = (dates[i]-dates[i-1]).days
        if  delta != 1:    
            for j in range(1,delta):
                tempday = dates[i-1] + pd.Timedelta(days=j)
                daystr = tempday.strftime('%Y/%m/%d')
                missing_day.append(daystr)         
    return  missing_day    
def day_before(missing_day):
    dates = pd.to_datetime(missing_day)
    day_before = [ (date - pd.Timedelta(days=1)).strftime('%Y/%m/%d') for date in dates]
    return  day_before
def day_after(missing_day):
    dates = pd.to_datetime(missing_day)
    day_after = [ (date + pd.Timedelta(days=1)).strftime('%Y/%m/%d') for date in dates]
    return  day_after
def fill_Missing_days1(ser):
    # ser here is in df form
    dates = ser.index
    missing_day = detect_Missing_days(dates)
    if len(missing_day)==0:
           return ser
    before = day_before(missing_day)
    after  = day_after(missing_day)
    for day in missing_day:
        i = missing_day.index(day)
        try:
            ser[day] = ser[[before[i],after[i]]].mean()
        except:
            print('Can not apply fill_Missing_days1')
            return
    ser.sort_index(inplace=True)
    return ser

# forward value smooth
def forward_mean_smooth(lst, smooth_window=7):
    result = []  
    if smooth_window == -1:
        for i in range(len(lst)):
            temp = lst[:i+1]
            result.append(sum(temp)/(i+1))
    else:     
        for i in range(len(lst)):
            if i<smooth_window-1:
                temp = lst[:i+1]
                result.append(sum(temp)/(i+1))
            else:
                temp = lst[i-smooth_window+1:i+1]
                result.append(sum(temp)/(smooth_window))
    return result
def forward_exp_smooth(lst, smooth_window=-1, Lambda=0.2):
    result = []  
    lst = np.array(lst)
    if smooth_window == -1:
        for i in range(len(lst)):
            weight = (Lambda*(1-Lambda)**np.arange(i+1))[::-1]
            temp = lst[:i+1]*weight
            result.append(temp.sum())
    else: 
        weight = (Lambda*(1-Lambda)**np.arange(smooth_window))[::-1]
        for i in range(len(lst)):
            if i<smooth_window:
                temp = weight[-(i+1):]*lst[:i+1]
                result.append(temp.sum())
            else:
                temp = weight*lst[i-smooth_window+1:i+1]
                result.append(temp.sum())
    return result
def forward_mix_smooth(lst, smooth_window=-1, Lambda=0.2):
    result = []  
    lst = np.array(lst)
    if smooth_window == -1:
        for i in range(len(lst)):
            weight = (Lambda*(1-Lambda)**np.arange(i+1))[::-1]
            if weight.sum()>=0.7:
                temp = lst[:i+1]*weight
                result.append(temp.sum())
            else:
                temp = lst[:i+1]
                result.append(sum(temp)/(i+1))
    else: 
        weight = (Lambda*(1-Lambda)**np.arange(smooth_window))[::-1]
        if weight.sum()<0.85:
            raise ValueError('weight.sum()<0.85')
        for i in range(len(lst)):
            if i<smooth_window:
                wt = weight[-(i+1):]
                if wt.sum()>=0.7:
                    temp = lst[:i+1]*wt
                    result.append(temp.sum())
                else:
                    temp = lst[:i+1]
                    result.append(sum(temp)/(i+1))
            else:
                temp = weight*lst[i-smooth_window+1:i+1]
                result.append(temp.sum())
    return result

### Calculation

In [ ]:
# Word Count
DocNum   = 6 # 1,2,3,4,5,6
top_prop = 0

temp_counts = wordcounts[DocNum-1]
print(temp_counts[temp_counts['Prop%']>=top_prop]['Prop%'].sum())
print(temp_counts[temp_counts['Prop%']>=top_prop]['Count'].sum())
print()
top = display_wordcounts('top', -1, temp_counts[temp_counts['Prop%']>=top_prop])

In [ ]:
%%time
# 7 Days
tfidfs_c7 = []
daily_tfidfs_c7 = pd.DataFrame(columns=['all_full','all_VN','all_N',
                                        'ori_full','ori_VN','ori_N'],
                               index=Days)
min_dfs = [0,0.0014941,0.0016394,0.0053368,0.0069746,0.0076622]

for i in range(6):
    Doc = Docs[i]
    print('Start Doc_%d'%(i+1))
    tfidf, daily_tfidf = track_tfidf(Doc, cal_window=7, min_df=min_dfs[i])
    tfidfs_c7.append(tfidf)
    daily_tfidfs_c7.iloc[:,i] = daily_tfidf.iloc[:,0]
    print('Finish Doc_%d'%(i+1))
    print('')

In [ ]:
# save data
daily_tfidfs_c7.to_csv('tw_pitt_daily_tfidf_raw_c7.csv', index=True)  

nameset = ['all_full','all_VN','all_N', 'ori_full','ori_VN','ori_N']
for i in range(6):
    df = tfidfs_c7[i]
    namstr = 'Doc%d_'%(i+1)+'tw_pitt_tfidf_withna_'+nameset[i]+'_c7.csv'
    df.to_csv(namstr, index=True)  

In [ ]:
# smooth
daily_tfidfs_mean_c7 = daily_tfidfs_c7.apply(lambda x: forward_mean_smooth(list(x), smooth_window=7), axis=0)
daily_tfidfs_exp_c7  = daily_tfidfs_c7.apply(lambda x: forward_exp_smooth(list(x), smooth_window=-1, Lambda=0.2), axis=0)
daily_tfidfs_mix_c7  = daily_tfidfs_c7.apply(lambda x: forward_mix_smooth(list(x), smooth_window=-1, Lambda=0.2), axis=0)

daily_tfidfs_mean_c7.to_csv('tw_pitt_daily_tfidf_mean_c7_s7.csv', index=True) 
daily_tfidfs_exp_c7.to_csv('tw_pitt_daily_tfidf_exp_c7_s-1.csv', index=True) 
daily_tfidfs_mix_c7.to_csv('tw_pitt_daily_tfidf_mix_c7_s-1.csv', index=True) 

### Visualization

In [ ]:
fig = plot_combined_tfidf(daily_tfidfs_c7)

In [ ]:
fig = plot_combined_tfidf(daily_tfidfs_mean_c7)
if False:
    fig.savefig('./plot/daily_tfidfs_mean_c7_s7.jpg', bbox_inches = 'tight')

In [ ]:
fig = plot_combined_tfidf(daily_tfidfs_exp_c7)
if True:
    fig.savefig('./plot/daily_tfidfs_exp_c7_s-1.jpg', bbox_inches = 'tight')

In [ ]:
fig = plot_combined_tfidf(daily_tfidfs_mix_c7)
if True:
    fig.savefig('./plot/daily_tfidfs_mix_c7_s-1.jpg', bbox_inches = 'tight')

## More Visualization

In [ ]:
# Load data
cal_window = 7

tfidfs = []
nameset = ['all_full','all_VN','all_N', 'ori_full','ori_VN','ori_N']
os.chdir(workspace('tweets'))  
for i in range(6):
    namstr = 'Doc%d_'%(i+1)+'tw_pitt_tfidf_withna_'+nameset[i]+'_c%d.csv'%cal_window
    df = pd.read_csv(namstr,index_col='Date')
    tfidfs.append(df)

### Functions 

In [ ]:
# Plot the tf-idf trendings for top 100 words with a maximum numbers of non-zero values
def trendingheatmap(df,topNumber):
    top_index = df.mean().sort_values(ascending = False)[:topNumber].index
    top_statistics = df.loc[:,top_index]
    maxVal = top_statistics.max().max()
    minVal = top_statistics.min().min()
    fig = plt.figure(figsize = (10,10))
    sns.heatmap(top_statistics.T, center = (maxVal+minVal)/2 , cmap = "rocket_r")
    return fig


# Average tf-idfs for each term
def plot_wordsAverage_tfidf( df, topNumber = 50 ):
    average = df.mean().sort_values( ascending = False )
    
    # Parameters
    labelsize = 20
    titlesize = 25
    ticksize = 20
    tickrotation = 90
    
    fig = plt.figure( figsize = (20, 10) ) 
    ser = average[:topNumber]

    x = np.arange(len(ser))
    bar_width = 0.5

    # Plot bar chart
    plt.rc('axes', axisbelow=True) # grid behind plot
    plt.grid(linestyle = '-', linewidth = 1)
    plt.bar(x , ser , label = 'Standardized Average TF-IDFs' , 
            width = bar_width , color = (0, 53/255, 102/255))


    # Add title & ticks
    plt.title('Word-wise Average TF-IDFs: Top %d'%topNumber, fontproperties = 'Times New Roman', 
              fontsize = titlesize, fontweight="bold");
    plt.xticks(x , ser.index, fontproperties = 'Times New Roman', fontsize = ticksize, 
               rotation = tickrotation, fontweight="bold");
    plt.yticks(fontproperties = 'Times New Roman', size = ticksize, fontweight="bold");

    return fig

### Display

In [ ]:
fig = trendingheatmap(tfidfs[DocNum - 1], 20)
